In [ ]:
# ==============================================================================
# КЛЕТКА 1. Конфигурация путей, модели и параметров вычислительного ядра (v3.2)
# ==============================================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
ONTOLOGY_TTL_PATH = PROJECT_ROOT / "data" / "aquabit_hydraulic_ontology_v3.ttl"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Целевая папка с выгрузкой CSV из ПК «Аквабит»
TARGET_CITY_OR_PATH = "модель семей"
GEOMETRY_INP_PATH = None  # Опционально: путь к .inp файлу, если координаты в отдельном файле

# Детализация временной шкалы на карте (48 шагов = интервал 30 мин в формате ЧЧ:ММ без меток R360)
TIMELINE_BUCKETS = 48

# Выбор локальной GGUF-модели для ИИ-диагностики:
# Вариант А (рекомендуется для однозначных причинно-следственных выводов без шаблонных повторов):
#   GGUF_REPO = "bartowski/Qwen2.5-7B-Instruct-GGUF"
#   GGUF_FILENAME = "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
# Вариант Б (облегченная модель 4B):
#   GGUF_REPO = "unsloth/Qwen3.5-4B-GGUF"
#   GGUF_FILENAME = "Qwen3.5-4B-Q4_K_M.gguf"
MODEL_PRESET = "Qwen2.5-7B-Instruct"  # "Qwen2.5-7B-Instruct" или "Qwen3.5-4B"

if MODEL_PRESET == "Qwen2.5-7B-Instruct":
    GGUF_REPO = "bartowski/Qwen2.5-7B-Instruct-GGUF"
    GGUF_FILENAME = "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
else:
    GGUF_REPO = "unsloth/Qwen3.5-4B-GGUF"
    GGUF_FILENAME = "Qwen3.5-4B-Q4_K_M.gguf"

LLAMA_SERVER_PATH = None  # Автопоиск llama-server в models/ или tools/
LLAMA_BACKEND = "cuda"    # "cuda" или "cpu"
LLAMA_N_GPU_LAYERS = 99
LLAMA_CONTEXT = 16384
INFERENCE_MAX_NEW_TOKENS = 4096

# Оставлять ли llama-server запущенным на порту 8080 после генерации отчета
# (позволяет кнопкам «Сгенерировать развернутый анализ графика (ИИ)» в HTML-отчете обращаться к живой модели)
KEEP_LLAMA_SERVER_FOR_HTML = True
LLAMA_HTML_PORT = 8080

print(f"Конфигурация v3.2 загружена. Модель: {GGUF_FILENAME} | Шагов шкалы времени: {TIMELINE_BUCKETS}")


In [ ]:
# ==============================================================================
# КЛЕТКА 2. Аналитическое ядро Аквабит v3.2, онтология и генератор отчетов
# ==============================================================================
import html
import importlib
import io
import math
import os
import re
import time
from typing import Any
import numpy as np
import pandas as pd

try:
    import aquabit_ai.ontology_engine as _oe
    import aquabit_ai.v2_engine as _ve
    import aquabit_ai.v2_report_and_align as _vr
    importlib.reload(_oe)
    importlib.reload(_ve)
    importlib.reload(_vr)
    from aquabit_ai.v2_report_and_align import render_standalone_html_report, generate_reference_pure_facts_report
    from aquabit_ai.ontology_engine import (
        load_hydraulic_ontology,
        evaluate_ontology_rules,
        analyze_problem_domains,
        build_enriched_ai_context,
        build_ontology_ai_system_prompt,
        extract_ai_dashboards_and_content,
    )
    from aquabit_ai.v2_engine import (
        process_aquabit_city_fast,
        process_aquabit_city_v3,
        parse_inp_metadata_and_topology,
        read_aquabit_csv,
        find_col,
    )
except ImportError:
    import ontology_engine as _oe
    import v2_engine as _ve
    import v2_report_and_align as _vr
    importlib.reload(_oe)
    importlib.reload(_ve)
    importlib.reload(_vr)
    from v2_report_and_align import render_standalone_html_report, generate_reference_pure_facts_report
    from ontology_engine import (
        load_hydraulic_ontology,
        evaluate_ontology_rules,
        analyze_problem_domains,
        build_enriched_ai_context,
        build_ontology_ai_system_prompt,
        extract_ai_dashboards_and_content,
    )
    from v2_engine import (
        process_aquabit_city_fast,
        process_aquabit_city_v3,
        parse_inp_metadata_and_topology,
        read_aquabit_csv,
        find_col,
    )

print("Аналитическое ядро Аквабит v3.2 (с графовой диагностикой соседних объектов) и онтологический движок готовы к работе.")


In [ ]:
# ==============================================================================
# КЛЕТКА 3. Генератор интерактивного HTML5-отчета v3.2
# ==============================================================================
from pathlib import Path
from typing import Any

def render_aquabit_html_v3(
    facts: dict[str, Any],
    charts: dict[str, Any],
    main_md: str,
    parsed_dashboards: dict[str, Any],
    out_html: Path,
    gen_sec: float = 0.0,
) -> str:
    """Собирает автономный интерактивный HTML5-отчет v3.2:
    - Сортировка и фильтрация во всех 4 таблицах + автофильтры из карточек уведомлений и текста ИИ (filter:tab:preset)
    - Сквозная связка График -> Таблица -> Карта (с пульсацией выбранного объекта)
    - Исправленный зум карты от центра экрана без серого экрана
    - Детальная шкала времени в формате ЧЧ:ММ без (R360) с мгновенным обновлением паспорта выбранного объекта
    - Живой контекстный ИИ-анализ графиков и выбранных участков таблиц по кнопке.
    """
    return render_standalone_html_report(
        facts=facts,
        chart_payload=charts,
        ai_markdown=main_md,
        output_html_path=out_html,
        generation_meta={"inference_sec": gen_sec},
        selected_dashboards_spec=parsed_dashboards,
    )

print("Функция сборки интерактивного отчета render_aquabit_html_v3 (v3.2) инициализирована.")


In [ ]:
# ==============================================================================
# КЛЕТКА 4. Запуск сквозного расчета: граф-диагностика, TTL-онтология и генерация отчета
# ==============================================================================
import socket
import subprocess
import tempfile
import time
import json
import re
import requests
from pathlib import Path
from typing import Any
from huggingface_hub import hf_hub_download
from IPython.display import display, HTML

_ACTIVE_LLAMA_PROC = None

def run_pipeline_v3(city_or_path: str | Path):
    global _ACTIVE_LLAMA_PROC
    t0 = time.perf_counter()
    ontology = load_hydraulic_ontology(ONTOLOGY_TTL_PATH)
    city_dir = Path(city_or_path) if Path(city_or_path).exists() else (RAW_DATA_DIR / str(city_or_path))
    inp_file = Path(GEOMETRY_INP_PATH) if GEOMETRY_INP_PATH else None

    print(f"[1/3] Обработка данных, трассировка графа смежности и оценка правил TTL-онтологии для «{city_dir.name}»...")
    facts, charts = process_aquabit_city_v3(
        city_dir,
        inp_path=inp_file,
        timeline_buckets=globals().get("TIMELINE_BUCKETS", 48),
    )
    facts["chart_payload"] = charts

    triggered_rules = evaluate_ontology_rules(facts, ontology)
    problem_domains = analyze_problem_domains(facts, triggered_rules, ontology=ontology)

    print(f"Ключевая проблемная область сети: «{problem_domains['primary_domain']}»")
    for d in problem_domains["ranked_domains"]:
        print(f"   • [{d['score']:.1f} б.] {d['title']} ({d['details']})")

    # Обогащенный причинно-следственный контекст с конкретными ID (Источник -> Насос -> Коллектор -> Труба)
    enriched_context = build_enriched_ai_context(facts, problem_domains["ranked_domains"], triggered_rules)

    print("[2/3] Формирование однозначного аналитического заключения ИИ по графу смежности...")
    system_prompt = build_ontology_ai_system_prompt(ontology)
    user_prompt = (
        f"Выполни глубокий технический аудит гидравлической модели города {facts['city_name']} "
        "на основе предрассчитанных причинно-следственных цепочек графа (Источник -> Насос -> Коллектор -> Отходящая труба). "
        "Обязательно используй кликабельные ссылки #ID и фильтр-ссылки на таблицы вида (см. [описание](filter:tab:preset)):\n"
        + json.dumps(enriched_context, ensure_ascii=False, indent=2)
    )

    raw_ai_text = ""
    gen_sec = 0.0

    exe_candidates = [
        Path(LLAMA_SERVER_PATH) if LLAMA_SERVER_PATH else None,
        MODELS_DIR / "llama-server.exe",
        PROJECT_ROOT / "llama-server.exe",
        PROJECT_ROOT / "tools" / "llama-server.exe",
        MODELS_DIR / "llama-server",
        PROJECT_ROOT / "llama-server",
    ]
    exe = next((p for p in exe_candidates if p and p.exists()), None)
    if exe:
        try:
            model_file = Path(
                hf_hub_download(
                    repo_id=GGUF_REPO,
                    filename=GGUF_FILENAME,
                    local_dir=str(MODELS_DIR / "gguf" / GGUF_REPO.replace("/", "--"))
                )
            )
            keep_alive = bool(globals().get("KEEP_LLAMA_SERVER_FOR_HTML", True))
            port = int(globals().get("LLAMA_HTML_PORT", 8080)) if keep_alive else 0
            if port == 0:
                with socket.socket() as s:
                    s.bind(("127.0.0.1", 0))
                    port = s.getsockname()[1]
            base_url = f"http://127.0.0.1:{port}"
            ngl_val = LLAMA_N_GPU_LAYERS if LLAMA_BACKEND.lower() == "cuda" else 0

            # Проверяем, не запущен ли уже сервер на этом порту
            already_running = False
            try:
                if requests.get(base_url + "/health", timeout=1).status_code == 200:
                    already_running = True
            except requests.RequestException:
                pass

            proc = None
            if not already_running:
                proc = subprocess.Popen(
                    [
                        str(exe), "-m", str(model_file),
                        "--host", "127.0.0.1", "--port", str(port),
                        "-c", str(LLAMA_CONTEXT),
                        "-ngl", str(ngl_val),
                        "--jinja"
                    ],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL,
                    creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0)
                )
                _ACTIVE_LLAMA_PROC = proc

            try:
                ready = already_running
                for _ in range(120):
                    if ready:
                        break
                    if proc is not None and proc.poll() is not None:
                        break
                    try:
                        if requests.get(base_url + "/health", timeout=1).status_code == 200:
                            ready = True
                            break
                    except requests.RequestException:
                        pass
                    time.sleep(1.0)

                if ready:
                    t_llm = time.perf_counter()
                    http_resp = requests.post(
                        base_url + "/v1/chat/completions",
                        json={
                            "model": "local",
                            "messages": [
                                {"role": "system", "content": system_prompt},
                                {"role": "user", "content": user_prompt}
                            ],
                            "max_tokens": INFERENCE_MAX_NEW_TOKENS,
                            "temperature": 0.20,
                            "top_p": 0.90,
                            "repeat_penalty": 1.12,
                            "chat_template_kwargs": {"enable_thinking": False}
                        },
                        timeout=300
                    )
                    if http_resp.ok:
                        raw_ai_text = http_resp.json()["choices"][0]["message"]["content"]
                        gen_sec = round(time.perf_counter() - t_llm, 2)
            finally:
                if proc is not None and not keep_alive:
                    proc.terminate()
                    try: proc.wait(timeout=5)
                    except subprocess.TimeoutExpired: proc.kill()
                elif keep_alive and ready:
                    print(f"🟢 Локальный ИИ-сервер оставлен активным на {base_url} для живого анализа графиков и таблиц по кнопке в HTML-отчете.")
        except Exception as e:
            print(f"Локальный llama-server недоступен ({e}), используется эталонный причинно-следственный генератор по графу.")

    if not raw_ai_text:
        t_ref = time.perf_counter()
        raw_ai_text = generate_reference_pure_facts_report(
            facts,
            ranked_domains=problem_domains["ranked_domains"],
            ontology=ontology,
        )
        gen_sec = round(time.perf_counter() - t_ref, 2)

    print("[3/3] Сборка интерактивного HTML5-паспорта сети v3.2...")
    main_md, parsed_dashboards = extract_ai_dashboards_and_content(raw_ai_text)
    selected_dashboards = parsed_dashboards.get("section_order") or [d["id"] for d in parsed_dashboards.get("dashboards", [])]
    print(f"ИИ определил ключевые приоритеты анализа: {selected_dashboards}")

    out_html = REPORTS_DIR / f"EPANET_AI_v3_{facts['city_name']}.html"
    render_aquabit_html_v3(facts, charts, main_md, parsed_dashboards, out_html, gen_sec)
    total_sec = round(time.perf_counter() - t0, 2)
    print(f"Отчет успешно сформирован за {total_sec} сек (ИИ: {gen_sec} сек).")
    print(f"Путь к HTML-отчету: {out_html.resolve()}")

    rel_path = f"reports/{out_html.name}"
    display(HTML(f'''
    <div style="background:#f8fafc;border:1px solid #cbd5e1;border-radius:10px;padding:16px 20px;margin-top:14px;box-shadow:0 2px 8px rgba(0,0,0,0.04);font-family:sans-serif;">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;">
            <span style="font-weight:700;font-size:15px;color:#0f172a;">Интерактивный отчет ПК «Аквабит» v3.2 ({facts['city_name']})</span>
            <a href="{rel_path}" target="_blank" style="display:inline-block;background:#0284c7;color:#fff;text-decoration:none;padding:7px 16px;border-radius:6px;font-weight:600;font-size:13px;">Открыть отчет в новой вкладке ↗</a>
        </div>
        <p style="margin:0 0 10px;font-size:13px;color:#64748b;">Встроены сортировка и фильтрация таблиц, сквозной переход График → Таблица → Карта, пульсация выбранного объекта, детальная шкала ЧЧ:ММ и живой ИИ-анализ по кнопке.</p>
        <iframe src="{rel_path}" style="width:100%;height:740px;border:1px solid #e2e8f0;border-radius:8px;background:#fff;" frameborder="0"></iframe>
    </div>
    '''))

run_pipeline_v3(TARGET_CITY_OR_PATH)


In [ ]:
# ==============================================================================
# КЛЕТКА 5. Управление фоновым llama-server для живого ИИ-диалога в HTML-отчете
# ==============================================================================
def stop_background_llama_server():
    global _ACTIVE_LLAMA_PROC
    if '_ACTIVE_LLAMA_PROC' in globals() and _ACTIVE_LLAMA_PROC is not None:
        _ACTIVE_LLAMA_PROC.terminate()
        _ACTIVE_LLAMA_PROC = None
        print("🛑 Фоновый llama-server остановлен.")
    else:
        print("ℹ️ Фоновый llama-server сейчас не запущен (HTML-отчет работает в автономном режиме предрассчитанного ИИ-контекста).")

# Раскомментируйте строку ниже, если нужно освободить видеопамять GPU после работы с HTML-отчетом:
# stop_background_llama_server()
